# Multi-page QA with InternVL2.5-8B (Kaggle)

Generates questions that can only be answered by combining **two or more
pages** of the same document:

* **Tabel Terpotong** – a table that continues on the next page;
* **Gabungan Grafik** – numbers from charts on different pages;
* **Teks Lintas Halaman** – facts stated in text on different pages;
* **Campuran** – text on one page explained by a table/chart on another.

How to use:
1. Settings → Accelerator **GPU T4 x2**, Internet **on** (or add the model
   from Kaggle Models and point `MODEL_PATH` at it).
2. Add your page images as a Dataset (file names `<document>_pNNN.png`).
3. Run cells 1–2 **once** (install + load model).
4. Edit and re-run cells 3–6 as often as you like; the model stays loaded.

Output records use the same format as `multi_internvl_fixed.py`, so the
review app (`main_multipage.py`) can read them.

## 1. Install (run once)
InternVL's remote code breaks on transformers >= 4.50 (and 5.x), so pin an
older one. Kaggle preinstalls a newer transformers: after this cell finishes,
**Run → Restart & clear cell outputs**, then continue from cell 2 (do not
re-run this cell).

In [ ]:
# sentencepiece 0.2.2 rejects InternLM2's tokenizer.model ("piece must not
# include null character") and transformers 4.46 hides that as a tokenizer of
# False; 0.2.1 loads it and has a Python 3.13 wheel.
!pip install "transformers==4.46.3" "tokenizers>=0.20,<0.21" "sentencepiece==0.2.1" timm einops accelerate bitsandbytes
!pip show transformers | grep -i ^version

## 2. Load the model (run once)
Guarded by `if "model" not in globals()`, so re-running this cell does not
reload. To switch model/precision, restart the kernel.

In [ ]:
import math

import torch
import transformers
from transformers import AutoConfig, AutoModel, AutoTokenizer

if tuple(int(x) for x in transformers.__version__.split(".")[:2]) >= (4, 50):
    raise RuntimeError(
        f"transformers {transformers.__version__} is loaded, but InternVL2.5 "
        "needs < 4.50. Run cell 1, then Run → Restart, then run this cell again."
    )

MODEL_PATH = "OpenGVLab/InternVL2_5-8B"
# One 16 GB GPU (P100 / single T4): set True. T4 x2: keep False, the model is
# split over both GPUs in float16 (T4 and P100 have no bfloat16).
LOAD_IN_8BIT = False


def split_model(path):
    """Device map from the InternVL model card: ViT + embeddings on GPU 0,
    language layers spread over all GPUs (GPU 0 counts as half)."""
    world_size = torch.cuda.device_count()
    config = AutoConfig.from_pretrained(path, trust_remote_code=True)
    num_layers = config.llm_config.num_hidden_layers
    per_gpu = math.ceil(num_layers / (world_size - 0.5))
    per_gpu = [per_gpu] * world_size
    per_gpu[0] = math.ceil(per_gpu[0] * 0.5)
    device_map, layer = {}, 0
    for gpu, count in enumerate(per_gpu):
        for _ in range(count):
            if layer < num_layers:
                device_map[f"language_model.model.layers.{layer}"] = gpu
                layer += 1
    for name in (
        "vision_model",
        "mlp1",
        "language_model.model.tok_embeddings",
        "language_model.model.embed_tokens",
        "language_model.output",
        "language_model.model.norm",
        "language_model.model.rotary_emb",
        "language_model.lm_head",
        f"language_model.model.layers.{num_layers - 1}",
    ):
        device_map[name] = 0
    return device_map


# Distinct names (vl_model / vl_tokenizer) so other cells or snippets that use
# "model" or "tokenizer" cannot overwrite them.
if not hasattr(globals().get("vl_model"), "chat"):
    print(f"GPUs: {torch.cuda.device_count()}")
    if LOAD_IN_8BIT:
        load_kwargs = {"load_in_8bit": True, "device_map": "auto"}
    elif torch.cuda.device_count() > 1:
        load_kwargs = {"device_map": split_model(MODEL_PATH)}
    else:
        load_kwargs = {"device_map": {"": 0}}
    vl_model = AutoModel.from_pretrained(
        MODEL_PATH,
        torch_dtype=torch.float16,
        low_cpu_mem_usage=True,
        use_flash_attn=False,
        trust_remote_code=True,
        **load_kwargs,
    ).eval()
    print("Model loaded.")
else:
    print("Model already loaded; skipping.")

if not hasattr(globals().get("vl_tokenizer"), "convert_tokens_to_ids"):
    vl_tokenizer = AutoTokenizer.from_pretrained(
        MODEL_PATH, trust_remote_code=True, use_fast=False
    )
if not hasattr(vl_tokenizer, "convert_tokens_to_ids"):
    raise RuntimeError(
        f"Tokenizer loaded as {vl_tokenizer!r}. Usually sentencepiece is too "
        "new: run `!pip install sentencepiece==0.2.1`, restart, run cell 2."
    )
print(f"Tokenizer: {type(vl_tokenizer).__name__}")

## 3. Settings (edit freely, then re-run cells 3–6)

In [ ]:
from pathlib import Path

# Folder with page images; searched recursively.
IMAGE_DIR = Path("/kaggle/input/bi-multi")
# Optional groups file ("<document>: 14-15 23-25"), same format as
# bi_multi_groups.txt. None = automatic windows of nearby pages.
GROUPS_FILE = None  # e.g. Path("/kaggle/input/bi-multi/bi_multi_groups.txt")
OUT_PATH = Path("/kaggle/working/bi_multipage_internvl_kaggle.jsonl")

GROUP_SIZE = 3          # pages per automatic window (2 or 3)
MAX_PAGE_GAP = 2        # pages further apart are never grouped
MAX_TILES = 6           # 448px tiles per page; 256 tokens each (+1 thumbnail)
N_QUESTIONS = 3         # questions per page group
ATTEMPTS = 4            # tries per group; accepted questions are kept
MAX_NEW_TOKENS = 1536
RETRY_TEMPERATURE = 0.5
LIMIT = None            # process only the first N groups (None = all)
KEEP_EVIDENCE = True    # save the per-page quotes in the output

MULTIPAGE_TYPES = {
    "Tabel Terpotong": (
        "tabel yang terpotong dan berlanjut ke halaman berikutnya; jawaban "
        "memerlukan baris/kolom dari bagian tabel di kedua halaman"
    ),
    "Gabungan Grafik": (
        "angka dari grafik di halaman berbeda digabung, dibandingkan, atau "
        "dihitung bersama"
    ),
    "Teks Lintas Halaman": (
        "fakta dari paragraf teks di halaman berbeda digabung untuk satu "
        "jawaban"
    ),
    "Campuran": (
        "penjelasan teks di satu halaman dikaitkan dengan angka tabel/grafik "
        "di halaman lain"
    ),
}
CATEGORIES = [
    "Text-Text Inference",
    "Chart-Information Extraction",
    "Chart-Numerical Calculation",
    "Chart-Time-Sensitive",
    "Table-Numerical Calculation",
    "Table-Comparison and Sorting",
]
BANNED_PHRASES = (
    "tabel ini", "tabel di atas", "grafik ini", "gambar ini", "teks ini",
    "teks di atas", "dokumen ini", "halaman ini", "berdasarkan tabel",
    "berdasarkan grafik", "menurut data di atas",
)
MIN_QUERY_WORDS = 8
MIN_QUOTE_CHARS = 8
MAX_QUOTE_SIMILARITY = 0.9
DUP_THRESHOLD = 0.85

PROMPT = """
Anda diberikan {n_pages} gambar halaman dari SATU dokumen yang sama, diberi
label [Halaman N]. Nomor halaman yang sah hanya: {page_list}. Gunakan nomor
label tersebut, BUKAN nomor yang tercetak di gambar.

Buat TEPAT {n_questions} pertanyaan dan jawaban dalam Bahasa Indonesia.

SYARAT LINTAS HALAMAN (paling penting): setiap pertanyaan WAJIB memakai
informasi dari MINIMAL DUA halaman berbeda dan TIDAK BOLEH bisa dijawab dari
satu halaman saja. Setiap halaman harus menyumbang informasi yang BERBEDA.
Pilih jenis lintas halaman yang paling cocok dengan isi halaman:
{type_block}

ATURAN:
1. Setiap pertanyaan memakai kategori yang BERBEDA.
2. Tanyakan fakta nyata dari isi dokumen, bukan elemen visual atau tata letak.
3. Jangan gunakan frasa: {banned}, nomor tabel, atau nomor gambar.
4. Sebutkan wilayah, periode, sektor, indikator, atau institusi yang dibahas.
   Pertanyaan minimal {min_words} kata.
5. Jangan mengarang angka. Salin angka persis; perhitungan hanya memakai angka
   yang terlihat, tulis langkah singkatnya di jawaban.
6. "answer_type": "pendek" untuk jawaban singkat, "panjang" bila perlu
   penjelasan.
7. Isi "evidence" LEBIH DULU: satu kutipan singkat dan persis per halaman yang
   dipakai (angka beserta nama baris/kolom atau periodenya). Kutipan antar
   halaman harus berbeda. "source_pages" = halaman-halaman pada evidence.

Kategori yang boleh dipakai:
{category_block}
{avoid_block}{feedback_block}
Keluarkan HANYA JSON valid, tanpa Markdown atau ```json. Format:
{{"result": [
  {{"query": "...", "category": "...", "multipage_type": "...",
    "evidence": [{{"page": {p1}, "quote": "..."}}, {{"page": {p2}, "quote": "..."}}],
    "answer": "...", "answer_type": "pendek", "source_pages": [{p1}, {p2}]}}
]}}
""".strip()

## 4. Helpers (re-run after editing)

In [ ]:
import difflib
import itertools
import json
import re
import time
import urllib.parse
from collections import defaultdict

import torchvision.transforms as T
from PIL import Image
from torchvision.transforms.functional import InterpolationMode

IMAGE_EXTS = {".png", ".jpg", ".jpeg", ".webp"}
PAGE_PATTERN = re.compile(r"^(?P<doc>.+)_p(?P<page>\d+)$", re.IGNORECASE)
INPUT_SIZE = 448
STOPWORDS = frozenset(
    """bagaimana berapa apa apakah mengapa kenapa siapa kapan dimana mana
    yang dan di ke dari pada dalam dengan untuk atau serta oleh adalah ini
    itu tersebut dibandingkan membandingkan perbandingan bandingkan
    berdasarkan menurut periode triwulan tahun sebesar secara terhadap
    antara kepada sebagai akan telah sudah lebih dapat bisa jelaskan
    sebutkan hitung tentukan""".split()
)

# ---------- image preprocessing (from the InternVL model card) ----------

TRANSFORM = T.Compose([
    T.Lambda(lambda img: img.convert("RGB")),
    T.Resize((INPUT_SIZE, INPUT_SIZE), interpolation=InterpolationMode.BICUBIC),
    T.ToTensor(),
    T.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
])


def find_closest_aspect_ratio(aspect_ratio, target_ratios, width, height):
    best_diff, best = float("inf"), (1, 1)
    area = width * height
    for ratio in target_ratios:
        diff = abs(aspect_ratio - ratio[0] / ratio[1])
        if diff < best_diff:
            best_diff, best = diff, ratio
        elif diff == best_diff and area > 0.5 * INPUT_SIZE**2 * ratio[0] * ratio[1]:
            best = ratio
    return best


def dynamic_preprocess(image, max_num):
    width, height = image.size
    ratios = sorted(
        {
            (i, j)
            for n in range(1, max_num + 1)
            for i in range(1, n + 1)
            for j in range(1, n + 1)
            if i * j <= max_num
        },
        key=lambda r: r[0] * r[1],
    )
    cols, rows = find_closest_aspect_ratio(width / height, ratios, width, height)
    resized = image.resize((INPUT_SIZE * cols, INPUT_SIZE * rows))
    tiles = [
        resized.crop((
            (i % cols) * INPUT_SIZE,
            (i // cols) * INPUT_SIZE,
            (i % cols + 1) * INPUT_SIZE,
            (i // cols + 1) * INPUT_SIZE,
        ))
        for i in range(cols * rows)
    ]
    if len(tiles) > 1:
        tiles.append(image.resize((INPUT_SIZE, INPUT_SIZE)))
    return tiles


def load_page(path, max_num=MAX_TILES):
    with Image.open(path) as source:
        image = source.convert("RGB")
    return torch.stack([TRANSFORM(tile) for tile in dynamic_preprocess(image, max_num)])


# ---------- page groups ----------

def normalize_name(name):
    """Repair names where percent encoding lost its percent sign."""
    if name.count(" 20") >= 2:
        name = name.replace(" 20", " ")
    return re.sub(r"\s+", " ", urllib.parse.unquote(name)).strip()


def scan_documents(folder):
    docs = defaultdict(dict)
    for path in sorted(folder.rglob("*")):
        if path.suffix.lower() not in IMAGE_EXTS:
            continue
        match = PAGE_PATTERN.match(normalize_name(path.stem))
        if match:
            doc = match.group("doc").strip().rstrip("_").strip()
            docs[doc][int(match.group("page"))] = path
    return dict(docs)


def group_key(doc, pages):
    return f"{doc}::" + "-".join(f"p{page:03d}" for page in pages)


def auto_groups(docs):
    groups = []
    for doc, pages in docs.items():
        numbers = sorted(pages)
        runs = [[numbers[0]]]
        for page in numbers[1:]:
            if page - runs[-1][-1] <= MAX_PAGE_GAP:
                runs[-1].append(page)
            else:
                runs.append([page])
        for run in runs:
            if len(run) < 2:
                continue
            chunks = [run[i : i + GROUP_SIZE] for i in range(0, len(run), GROUP_SIZE)]
            if len(chunks) > 1 and len(chunks[-1]) == 1:
                chunks[-1] = run[-2:]
            groups += [(doc, chunk) for chunk in chunks]
    return groups


def parse_spec(spec):
    if re.fullmatch(r"\d+-\d+", spec):
        first, last = map(int, spec.split("-"))
        return list(range(first, last + 1))
    return sorted({int(part) for part in spec.split(",")})


def file_groups(docs, path):
    groups = []
    for line in Path(path).read_text(encoding="utf-8").splitlines():
        line = line.split("#", 1)[0].strip()
        if ":" not in line:
            continue
        fragment, specs = line.rsplit(":", 1)
        fragment = fragment.strip().lower()
        matches = [d for d in docs if d.lower() == fragment] or [
            d for d in docs if fragment in d.lower()
        ]
        if not matches:
            print(f"WARNING: no document matches {fragment!r}")
        for doc in matches:
            for spec in specs.replace(";", " ").split():
                pages = parse_spec(spec)
                missing = [p for p in pages if p not in docs[doc]]
                if missing or len(pages) < 2:
                    print(f"WARNING: skip {doc} {spec} (missing {missing})")
                    continue
                groups.append((doc, pages))
    return groups


# ---------- prompt ----------

def build_prompt(pages, n_questions, used_categories=(), avoid=(), feedback=None):
    image_block = "\n".join(f"[Halaman {page}]\n<image>" for page in pages)
    avoid_block = ""
    if avoid:
        avoid_block = (
            "\nPertanyaan berikut sudah ada. Jangan mengulang atau "
            "memparafrasakannya:\n" + "\n".join(f"- {q}" for q in avoid[-15:]) + "\n"
        )
    feedback_block = ""
    if feedback:
        feedback_block = (
            f"\nPERCOBAAN SEBELUMNYA DITOLAK karena: {feedback}\n"
            "Perbaiki kesalahan tersebut.\n"
        )
    text = PROMPT.format(
        n_pages=len(pages),
        page_list=", ".join(map(str, pages)),
        n_questions=n_questions,
        type_block="\n".join(f'- "{k}": {v}' for k, v in MULTIPAGE_TYPES.items()),
        banned=", ".join(f'"{p}"' for p in BANNED_PHRASES),
        min_words=MIN_QUERY_WORDS,
        category_block="\n".join(
            f"- {c}" for c in CATEGORIES if c not in used_categories
        ),
        avoid_block=avoid_block,
        feedback_block=feedback_block,
        p1=pages[0],
        p2=pages[1],
    )
    return f"{image_block}\n\n{text}"


def generate(pages_paths, question, temperature):
    tiles = [load_page(path) for path in pages_paths]
    pixel_values = torch.cat(tiles).to(torch.float16).cuda()
    config = {"max_new_tokens": MAX_NEW_TOKENS, "do_sample": temperature > 0}
    if temperature > 0:
        config.update(temperature=temperature, top_p=0.9)
    with torch.inference_mode():
        response = vl_model.chat(
            vl_tokenizer,
            pixel_values,
            question,
            generation_config=config,
            num_patches_list=[t.size(0) for t in tiles],
        )
    del pixel_values
    torch.cuda.empty_cache()
    return response


# ---------- parsing and validation ----------

def extract_json(text):
    text = re.sub(r"```(?:json)?", "", text or "", flags=re.I).strip()
    start = text.find("{")
    if start < 0:
        raise ValueError("no JSON object in output")
    depth, quoted, escaped = 0, False, False
    for index in range(start, len(text)):
        char = text[index]
        if escaped:
            escaped = False
        elif quoted and char == "\\":
            escaped = True
        elif char == '"':
            quoted = not quoted
        elif not quoted and char == "{":
            depth += 1
        elif not quoted and char == "}":
            depth -= 1
            if depth == 0:
                return json.loads(text[start : index + 1])
    raise ValueError("JSON output was truncated")


def normalize_text(text):
    return re.sub(r"\s+", " ", str(text).strip().lower())


def content_words(text):
    words = re.findall(r"[a-z0-9][a-z0-9.,%-]*", text.lower())
    return [w.strip(".,-") for w in words if w.strip(".,-") and w not in STOPWORDS]


def is_near_duplicate(query, others):
    words = content_words(query)
    for other in others:
        other_words = content_words(other)
        if not other_words:
            continue
        union = set(words) | set(other_words)
        if union and len(set(words) & set(other_words)) / len(union) >= 0.75:
            return True
        if difflib.SequenceMatcher(
            None, " ".join(words), " ".join(other_words)
        ).ratio() >= DUP_THRESHOLD:
            return True
    return False


def validate_item(item, pages):
    """Return an error message, or None when the question is acceptable."""
    if not isinstance(item, dict):
        return "item is not an object"
    for field in ("query", "category", "multipage_type", "evidence", "answer",
                  "answer_type", "source_pages"):
        if field not in item:
            return f"missing '{field}'"
    query = str(item["query"])
    if not str(item["answer"]).strip():
        return "empty answer"
    if len(query.split()) < MIN_QUERY_WORDS or len(content_words(query)) < 4:
        return f"query too generic: {query!r}"
    for phrase in BANNED_PHRASES:
        if phrase in query.lower():
            return f"query uses banned phrase '{phrase}'"
    if item["category"] not in CATEGORIES:
        return f"invalid category {item['category']!r}"
    if item["multipage_type"] not in MULTIPAGE_TYPES:
        return f"invalid multipage_type {item['multipage_type']!r}"
    if item["answer_type"] not in ("pendek", "panjang"):
        return "invalid answer_type"

    evidence = item["evidence"]
    if not isinstance(evidence, list):
        return "evidence must be a list of {page, quote}"
    quotes = {}
    for entry in evidence:
        if not isinstance(entry, dict):
            return "evidence entries must be objects"
        page, quote = entry.get("page"), str(entry.get("quote", "")).strip()
        if page not in pages:
            return f"evidence cites page {page}, valid pages are {pages}"
        if len(quote) >= MIN_QUOTE_CHARS:
            quotes[page] = normalize_text(quote)
    if len(quotes) < 2:
        return "evidence must quote at least two different pages"
    for a, b in itertools.combinations(quotes.values(), 2):
        if difflib.SequenceMatcher(None, a, b).ratio() >= MAX_QUOTE_SIMILARITY:
            return "evidence quotes from different pages are the same"
    source = item["source_pages"]
    if not isinstance(source, list) or set(source) != set(quotes):
        return f"source_pages {source} must equal evidence pages {sorted(quotes)}"
    return None


# ---------- one group ----------

def run_group(doc, pages, paths, seen_queries, verbose=True):
    """Generate N_QUESTIONS for one page group, retrying only rejected ones.
    Returns (record, accepted_items)."""
    kept, feedback, raw, error = [], None, "", None
    for attempt in range(1, ATTEMPTS + 1):
        needed = N_QUESTIONS - len(kept)
        prompt = build_prompt(
            pages,
            needed,
            used_categories={item["category"] for item in kept},
            avoid=seen_queries + [item["query"] for item in kept],
            feedback=feedback,
        )
        temperature = 0.0 if attempt == 1 else RETRY_TEMPERATURE
        # Generation errors other than OOM are bugs, not bad model output:
        # let them raise with a full traceback instead of retrying.
        try:
            raw = generate(paths, prompt, temperature)
        except torch.cuda.OutOfMemoryError:
            torch.cuda.empty_cache()
            error = "CUDA out of memory; lower MAX_TILES or GROUP_SIZE"
            break
        try:
            data = extract_json(raw)
            items = data.get("result", data) if isinstance(data, dict) else data
            if isinstance(items, dict):
                items = list(items.values())
            if not isinstance(items, list):
                raise ValueError("'result' must be a list")
        except Exception as exc:
            feedback = error = f"output was not valid JSON ({exc})"
            continue

        problems = []
        for item in items[:needed]:
            problem = validate_item(item, pages)
            if not problem and item["category"] in {k["category"] for k in kept}:
                problem = f"category '{item['category']}' already used"
            if not problem and is_near_duplicate(
                item["query"], seen_queries + [k["query"] for k in kept]
            ):
                problem = f"near-duplicate question: {item['query']}"
            if problem:
                problems.append(problem)
            else:
                kept.append(item)
        if len(items) < needed:
            problems.append(f"only {len(items)} of {needed} questions returned")
        if len(kept) >= N_QUESTIONS:
            error = None
            break
        feedback = error = "; ".join(problems)
        if verbose:
            print(f"  attempt {attempt}: {len(kept)}/{N_QUESTIONS} kept — {error}")

    record = {
        "group_key": group_key(doc, pages),
        "document": doc,
        "pages": pages,
        "source_paths": [str(p) for p in paths],
        "attempts": attempt,
    }
    if error is None:
        result = kept if KEEP_EVIDENCE else [
            {k: v for k, v in item.items() if k != "evidence"} for item in kept
        ]
        record["qa"] = {"result": result}
    else:
        record.update(
            qa=None,
            error=f"{error} (accepted {len(kept)}/{N_QUESTIONS})",
            raw_output=raw,
        )
    return record, kept


# ---------- build the work list ----------

def explain_no_groups():
    """Print why nothing was found, then stop."""
    if not IMAGE_DIR.exists():
        print(f"IMAGE_DIR does not exist: {IMAGE_DIR}\nFolders in /kaggle/input:")
        for path in sorted(Path("/kaggle/input").glob("*/*"))[:30]:
            print("  ", path)
    else:
        images = [p for p in IMAGE_DIR.rglob("*") if p.suffix.lower() in IMAGE_EXTS]
        print(f"{len(images)} image(s) under {IMAGE_DIR}. Examples:")
        for path in images[:5]:
            print("  ", path.name)
        if images and not docs:
            print("File names must look like '<document>_p014.png'.")
        elif docs and GROUPS_FILE:
            print("No line in GROUPS_FILE matched these documents:")
            for doc in list(docs)[:10]:
                print("  ", doc, sorted(docs[doc]))
        elif docs:
            print(f"No two pages of one document are within {MAX_PAGE_GAP} "
                  "pages of each other; raise MAX_PAGE_GAP or use GROUPS_FILE.")
    raise RuntimeError("No page groups found; fix IMAGE_DIR / GROUPS_FILE in cell 3.")


docs = scan_documents(IMAGE_DIR)
groups = file_groups(docs, GROUPS_FILE) if GROUPS_FILE else auto_groups(docs)
print(f"Documents: {len(docs)} | page groups: {len(groups)}")
if not groups:
    explain_no_groups()
for doc, pages in groups[:10]:
    print(f"  {doc[:70]}  pages {pages}")

## 5. Try one group (quick check of the prompt)
Change `TEST_INDEX` and re-run to look at another group. Nothing is saved.

In [ ]:
from IPython.display import display

TEST_INDEX = 0

doc, pages = groups[TEST_INDEX]
paths = [docs[doc][p] for p in pages]
for path in paths:
    with Image.open(path) as page_image:
        preview = page_image.copy()
    preview.thumbnail((500, 500))
    display(preview)

start = time.time()
record, _ = run_group(doc, pages, paths, seen_queries=[])
print(f"{time.time() - start:.1f}s")
print(json.dumps(record.get("qa") or record, ensure_ascii=False, indent=2)[:6000])

## 6. Run all groups
Appends to `OUT_PATH` and skips groups already done, so you can stop and
re-run this cell to resume. Delete the file to start over.

In [ ]:
done, seen_by_doc = set(), defaultdict(list)
if OUT_PATH.exists():
    for line in OUT_PATH.read_text(encoding="utf-8").splitlines():
        try:
            old = json.loads(line)
        except json.JSONDecodeError:
            continue
        if old.get("qa") and not old.get("error"):
            done.add(old["group_key"])
            seen_by_doc[old["document"]] += [q["query"] for q in old["qa"]["result"]]

todo = [(d, p) for d, p in groups if group_key(d, p) not in done]
if LIMIT is not None:
    todo = todo[:LIMIT]
print(f"Done: {len(done)} | todo: {len(todo)}")

ok = fail = 0
start = time.time()
with OUT_PATH.open("a", encoding="utf-8") as stream:
    for number, (doc, pages) in enumerate(todo, 1):
        paths = [docs[doc][p] for p in pages]
        record, kept = run_group(doc, pages, paths, seen_by_doc[doc])
        stream.write(json.dumps(record, ensure_ascii=False) + "\n")
        stream.flush()
        if record.get("qa"):
            ok += 1
            seen_by_doc[doc] += [item["query"] for item in kept]
            status = "OK  "
        else:
            fail += 1
            status = "FAIL"
        elapsed = time.time() - start
        print(
            f"[{number}/{len(todo)}] {status} {record['group_key'][:80]} | "
            f"ok={ok} fail={fail} | {elapsed / number:.0f}s/group"
        )
print(f"Finished in {time.time() - start:.0f}s → {OUT_PATH}")

## 7. Inspect results

In [ ]:
import pandas as pd

rows = []
for line in OUT_PATH.read_text(encoding="utf-8").splitlines():
    rec = json.loads(line)
    for item in (rec.get("qa") or {}).get("result", []):
        rows.append({
            "document": rec["document"][:50],
            "pages": rec["pages"],
            "type": item.get("multipage_type"),
            "category": item.get("category"),
            "query": item.get("query"),
            "answer": item.get("answer"),
            "source_pages": item.get("source_pages"),
        })
df = pd.DataFrame(rows)
print(df["type"].value_counts() if len(df) else "no questions yet")
df.head(20)